In [1]:
from pathlib import Path
import tensorflow as tf

DATASET_DIR = "Dataset"
IMAGE_SIZE = (264, 264)
BATCH_SIZE = 32
EPOCHS = 15
VALIDATION_SPLIT = 0.2
SEED = 42

tf.keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()


def numeric_sort_key(name: str):
    return int(name) if name.isdigit() else name


dataset_path = Path(DATASET_DIR)
class_names = sorted(
    [folder.name for folder in dataset_path.iterdir() if folder.is_dir()],
    key=numeric_sort_key,
)

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=VALIDATION_SPLIT,
    subset="training",
    seed=SEED,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    class_names=class_names,
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=VALIDATION_SPLIT,
    subset="validation",
    seed=SEED,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    class_names=class_names,
)

train_ds = train_ds.prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.prefetch(tf.data.AUTOTUNE)

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomRotation(0.02, seed=SEED),
    tf.keras.layers.RandomTranslation(0.03, 0.03, seed=SEED + 1),
])


model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(264, 264, 3)),
    data_augmentation,
    tf.keras.layers.Rescaling(1.0 / 255),
    tf.keras.layers.Conv2D(32, (3, 3), activation="relu"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(64, (3, 3), activation="relu"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(128, (3, 3), activation="relu"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(len(class_names), activation="softmax"),
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

# early_stopping = tf.keras.callbacks.EarlyStopping(
#     monitor="val_loss",
#     patience=3,
#     restore_best_weights=True
# )

model.summary()
history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS)

model.save("tamil_character_cnn.keras")
print("Classes:", class_names)
print("Model saved as tamil_character_cnn.keras")

Found 550 files belonging to 11 classes.
Using 440 files for training.
Found 550 files belonging to 11 classes.
Using 110 files for validation.


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sequential (Sequential)         │ (None, 264, 264, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling (Rescaling)           │ (None, 264, 264, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 262, 262, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 131, 131, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 129, 129, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 64, 64, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 62, 62, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 31, 31, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 123008)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │    15,745,152 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 11)             │         1,419 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 15,839,819 (60.42 MB)

 Trainable params: 15,839,819 (60.42 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/15
14/14 ━━━━━━━━━━━━━━━━━━━━ 21s 1s/step - accuracy: 0.1000 - loss: 2.4526 - val_accuracy: 0.0545 - val_loss: 2.3944
Epoch 2/15
14/14 ━━━━━━━━━━━━━━━━━━━━ 17s 1s/step - accuracy: 0.1364 - loss: 2.3675 - val_accuracy: 0.2000 - val_loss: 2.2599
Epoch 3/15
14/14 ━━━━━━━━━━━━━━━━━━━━ 16s 1s/step - accuracy: 0.2386 - loss: 2.1413 - val_accuracy: 0.2818 - val_loss: 2.0436
Epoch 4/15
14/14 ━━━━━━━━━━━━━━━━━━━━ 16s 1s/step - accuracy: 0.3636 - loss: 1.9352 - val_accuracy: 0.3364 - val_loss: 2.1412
Epoch 5/15
14/14 ━━━━━━━━━━━━━━━━━━━━ 16s 1s/step - accuracy: 0.4295 - loss: 1.7678 - val_accuracy: 0.3727 - val_loss: 2.0067
Epoch 6/15
14/14 ━━━━━━━━━━━━━━━━━━━━ 16s 1s/step - accuracy: 0.4455 - loss: 1.6037 - val_accuracy: 0.5182 - val_loss: 1.7038
Epoch 7/15
14/14 ━━━━━━━━━━━━━━━━━━━━ 16s 1s/step - accuracy: 0.5432 - loss: 1.4116 - val_accuracy: 0.5091 - val_loss: 1.5548
Epoch 8/15
14/14 ━━━━━━━━━━━━━━━━━━━━ 16s 1s/step - accuracy: 0.5932 - loss: 1.2387 - val_accuracy: 0.5000 - val_loss: